# Data Cleaning and Statistical Analysis

## First, we must import the necessary libraries

In [1]:
import pandas as pd
from pathlib import Path
import json

## Helper functions

`retrieve_songs` loads the raw iTunes search results and returns a DataFrame with `year`, `title`, `artist`, and `runtime` (in milliseconds, taken from the top iTunes search result for each song).

In [2]:
def retrieve_songs() -> pd.DataFrame:
    ITUNES_FILE = Path("../data/raw/itunes_tracks.json")
    with open(ITUNES_FILE, "r", encoding="utf-8") as f:
        itunes_tracks = json.load(f)

    rows = []
    for song in itunes_tracks:
        results = song["response"]["results"]

        # Take the runtime of the top iTunes search result (None if no results)
        runtime = results[0].get("trackTimeMillis") if results else None
        rows.append({
            "year": song["year"],
            "title": song["title"],
            "artist": song["artist"],
            "runtime": runtime,  # milliseconds
        })
    return pd.DataFrame(rows)

## Load the raw iTunes search results and display the first few rows of the DataFrame

In [3]:
df = retrieve_songs()
df

,year,title,artist,runtime
0,1970,"""Bridge Over Troubled Water""",Simon & Garfunkel,296462
1,1970,"""(They Long to Be) Close to You""",The Carpenters,276000
2,1971,"""Joy to the World""",Three Dog Night,195533
3,1971,"""Maggie May""/""Reason to Believe""",Rod Stewart,350147
4,1972,"""The First Time Ever I Saw Your Face""",Roberta Flack,259533
...,...,...,...,...
107,2023,"""Flowers""",Miley Cyrus,200600
108,2024,"""Lose Control""",Teddy Swims,210689
109,2024,"""A Bar Song (Tipsy)""",Shaboozey,163864
110,2025,"""Die with a Smile""",Lady Gaga and Bruno Mars,251668


## Step 1: Clean the data

### 1. Missing values

In [4]:
# count missing values in each column
missing_values_counts = df.isna().sum()
missing_values_counts

year       0
title      0
artist     0
runtime    0
dtype: int64

In [5]:
# missing value percentage for each column
missing_values_percentages = missing_values_counts / len(df) * 100
missing_values_percentages

year       0.0
title      0.0
artist     0.0
runtime    0.0
dtype: float64

In [6]:
# drop any songs with no runtime
df = df.dropna(subset=["runtime"])
len(df)

112

### 2. Formatting the titles and artists

The titles from Wikipedia still have quote marks around them, and double A-sides are listed as `"Song A"/"Song B"`. We remove the quotes, keep only the first song (the A-side), and strip any extra spaces.

In [7]:
# keep only the first song of a double A-side, e.g. '"Maggie May"/"Reason to Believe"'
df["title"] = df["title"].str.replace('" / "', '"/"')
df["title"] = df["title"].str.split('"/"').str[0]

# remove the quote marks and extra spaces
df["title"] = df["title"].str.strip('"').str.strip()
df["artist"] = df["artist"].str.strip()
df.head()

,year,title,artist,runtime
0,1970,Bridge Over Troubled Water,Simon & Garfunkel,296462
1,1970,(They Long to Be) Close to You,The Carpenters,276000
2,1971,Joy to the World,Three Dog Night,195533
3,1971,Maggie May,Rod Stewart,350147
4,1972,The First Time Ever I Saw Your Face,Roberta Flack,259533


In [8]:
# get list of unique values for the artist column to look for inconsistent names
artist_unique_values = sorted(df["artist"].unique())
artist_unique_values

['50 Cent',
 'Ace of Base',
 'Adele',
 'Alicia Keys',
 'All-4-One',
 'Andy Gibb',
 'Ashanti',
 'Barbra Streisand',
 'Bee Gees',
 'Beyoncé',
 'Blondie',
 'Bobby Brown',
 'Boyz II Men',
 'Brandy and Monica',
 'Bryan Adams',
 'Captain & Tennille',
 'Carly Rae Jepsen',
 'Cher',
 'Chicago',
 'Color Me Badd',
 'Coolio featuring L.V.',
 'Daniel Powter',
 'Diana Ross & Lionel Richie',
 'Dionne and Friends (Dionne Warwick, Gladys Knight, Elton John and Stevie Wonder)',
 'Donna Summer',
 'Drake',
 'Dua Lipa',
 'Ed Sheeran',
 'Elton John',
 'Elton John & Kiki Dee',
 'Faith Hill',
 'Flo Rida featuring T-Pain',
 'George Michael',
 "Gilbert O'Sullivan",
 'Glass Animals',
 'Glen Campbell',
 'Gotye featuring Kimbra',
 'Gwen Stefani',
 'Harry Styles',
 'Heart',
 'INXS',
 'Jewel',
 'Jim Croce',
 'Justin Bieber',
 'Katy Perry featuring Juicy J',
 'Kendrick Lamar and SZA',
 'Kesha',
 'Kim Carnes',
 'LMFAO featuring Lauren Bennett and GoonRock',
 'Lady Antebellum',
 'Lady Gaga',
 'Lady Gaga and Bruno Mars'

Some artists use "featuring" and some use "and", but each one is written the same way every time it appears, so no artist names need to be replaced.

### 3. Wrong values

The runtime comes from the top iTunes search result, so we checked each song against the raw iTunes data. Three runtimes are wrong:

- **"Another Brick in the Wall, Part II" (1980):** the top result was **Part 1** (3:11). The correct runtime of Part 2 from *The Wall* (3:59) is in the same iTunes results, so we replace it.
- **"You Were Meant for Me" (Jewel, 1997):** the top result is an unrelated song by a different artist.
- **"Poker Face" (Lady Gaga, 2009):** the top result is a remix; the original song isn't in the iTunes results.

There is no correct runtime for the last two, so we drop them.

In [9]:
# replace the runtime of Another Brick in the Wall with Part 2 (from The Wall, in the raw iTunes data)
df.loc[df["title"] == "Another Brick in the Wall, Part II", "runtime"] = 238947

# drop the two songs with no correct runtime
df = df[df["title"] != "You Were Meant for Me"]
df = df[df["title"] != "Poker Face"]
len(df)

110

### 4. Duplicates

In [10]:
# check if any song appears more than once
print("Duplicate songs:", df.duplicated(subset=["title", "artist"]).sum())

df = df.drop_duplicates(subset=["title", "artist"])

Duplicate songs: 0


### 5. Columns and data types

The runtime is in milliseconds, so we rename it and add columns in seconds and minutes. We also add a `decade` column, since the project compares decades.

In [11]:
df = df.rename(columns={"runtime": "runtime_ms"})
df["runtime_ms"] = df["runtime_ms"].astype(int)
df["runtime_sec"] = df["runtime_ms"] / 1000
df["runtime_min"] = df["runtime_sec"] / 60

df["decade"] = (df["year"] // 10) * 10

# move decade to the first column
decade_col = df.pop("decade")
df.insert(0, "decade", decade_col)

df = df.reset_index(drop=True)
df.dtypes

decade           int64
year             int64
title           object
artist             str
runtime_ms       int64
runtime_sec    float64
runtime_min    float64
dtype: object

### 6. Outliers

We use the IQR rule: a runtime is an outlier if it is more than 1.5 × IQR below Q1 or above Q3.

In [12]:
q1 = df["runtime_min"].quantile(0.25)
q3 = df["runtime_min"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
print("Lower limit:", round(lower, 2), "minutes")
print("Upper limit:", round(upper, 2), "minutes")

outliers = df[(df["runtime_min"] < lower) | (df["runtime_min"] > upper)]
outliers

Lower limit: 2.2 minutes
Upper limit: 5.68 minutes


,decade,year,title,artist,runtime_ms,runtime_sec,runtime_min
3,1970,1971,Maggie May,Rod Stewart,350147,350.147,5.835783
12,1970,1976,Silly Love Songs,Wings,354240,354.240,5.904000
28,1980,1984,When Doves Cry,Prince,354133,354.133,5.902217
42,1990,1991,(Everything I Do) I Do It for You,Bryan Adams,394133,394.133,6.568883
44,1990,1992,End of the Road,Boyz II Men,351467,351.467,5.857783


All 5 outliers are long songs (5.8 to 6.6 minutes): *Maggie May*, *Silly Love Songs*, *When Doves Cry*, *(Everything I Do) I Do It for You* and *End of the Road*. These are the real lengths of these songs, not data errors, and long songs being more common in the 70s–90s is part of what the project is measuring. So we **keep** them in the data.

### Cleaned data

In [13]:
df

,decade,year,title,artist,runtime_ms,runtime_sec,runtime_min
0,1970,1970,Bridge Over Troubled Water,Simon & Garfunkel,296462,296.462,4.941033
1,1970,1970,(They Long to Be) Close to You,The Carpenters,276000,276.000,4.600000
2,1970,1971,Joy to the World,Three Dog Night,195533,195.533,3.258883
3,1970,1971,Maggie May,Rod Stewart,350147,350.147,5.835783
4,1970,1972,The First Time Ever I Saw Your Face,Roberta Flack,259533,259.533,4.325550
...,...,...,...,...,...,...,...
105,2020,2023,Flowers,Miley Cyrus,200600,200.600,3.343333
106,2020,2024,Lose Control,Teddy Swims,210689,210.689,3.511483
107,2020,2024,A Bar Song (Tipsy),Shaboozey,163864,163.864,2.731067
108,2020,2025,Die with a Smile,Lady Gaga and Bruno Mars,251668,251.668,4.194467
